## Recommendation Chatbot with ChatGPT

This is a simple version of the Recommendation Chatbot discussed in class.
In this version, we manually input some samples for products and customer history. In a real system, these will be connected to existing databases. 
This sample is an updated version of a tutorial found at https://github.com/norahsakal/chatgpt-product-recommendation-embeddings

In [11]:
#import the basic packages and functions that we need to use
from openai import OpenAI
import pandas as pd
from scipy.spatial.distance import cosine

### 1. Setting up

In [12]:
#Input your API Key here, just like we did in Lecture 10
#If you do not have api_key, you can still follow the code and learn 
api_key = ""
client = OpenAI(api_key = api_key)

#we need to define two functions that are used throughout this exercise

#one is the get_embedding function which helps us connect to OPENAI and get an embedding
def get_embedding(text, model="text-embedding-ada-002"):
   text = text.replace("\n", " ")
   return client.embeddings.create(input = [text], model=model).data[0].embedding

#the second to use the cosine similarity we already seen in lecture 10. 
#This time we define our own cosine_similarity directly from the cosine distance
#the cosine function imported gives us the cosine distance, the similarity is 1-cosine distance
def cosine_similarity(x, y):
    return 1 - cosine(x,y)

### 2. Create product data

In [13]:
#define some products, this is to mimic our product database
product_data = [{
    "prod_id": 1,
    "prod": "moisturizer",
    "brand":"Aveeno",
    "description": "for dry skin"
},
{
    "prod_id": 2,
    "prod": "foundation",
    "brand":"Maybelline",
    "description": "medium coverage"
},
{
    "prod_id": 3,
    "prod": "moisturizer",
    "brand":"CeraVe",
    "description": "for dry skin"
},
{
    "prod_id": 4,
    "prod": "nail polish",
    "brand":"OPI",
    "description": "raspberry red"
},
{
    "prod_id": 5,
    "prod": "concealer",
    "brand":"chanel",
    "description": "medium coverage"
},
{
    "prod_id": 6,
    "prod": "moisturizer",
    "brand":"Ole Henkrisen",
    "description": "for oily skin"
},
{
    "prod_id": 7,
    "prod": "moisturizer",
    "brand":"CeraVe",
    "description": "for normal to dry skin"
},
{
    "prod_id": 8,
    "prod": "moisturizer",
    "brand":"First Aid Beauty",
    "description": "for dry skin"
},{
    "prod_id": 9,
    "prod": "makeup sponge",
    "brand":"Sephora",
    "description": "super-soft, exclusive, latex-free foam"
}]

In [14]:
#put the product into a pandas dataframe
product_data_df = pd.DataFrame(product_data)
#this is to show how it looks like in a dataframe format
product_data_df

,prod_id,prod,brand,description
0,1,moisturizer,Aveeno,for dry skin
1,2,foundation,Maybelline,medium coverage
2,3,moisturizer,CeraVe,for dry skin
3,4,nail polish,OPI,raspberry red
4,5,concealer,chanel,medium coverage
5,6,moisturizer,Ole Henkrisen,for oily skin
6,7,moisturizer,CeraVe,for normal to dry skin
7,8,moisturizer,First Aid Beauty,for dry skin
8,9,makeup sponge,Sephora,"super-soft, exclusive, latex-free foam"


In [15]:
#we add a column called "combined", this is essentially a concatenation of the values in each row into one.
#this will be used to generate our embeddings 
product_data_df['combined'] = product_data_df.apply(lambda row: f"{row['brand']}, {row['prod']}, {row['description']}", axis=1)
product_data_df

,prod_id,prod,brand,description,combined
0,1,moisturizer,Aveeno,for dry skin,"Aveeno, moisturizer, for dry skin"
1,2,foundation,Maybelline,medium coverage,"Maybelline, foundation, medium coverage"
2,3,moisturizer,CeraVe,for dry skin,"CeraVe, moisturizer, for dry skin"
3,4,nail polish,OPI,raspberry red,"OPI, nail polish, raspberry red"
4,5,concealer,chanel,medium coverage,"chanel, concealer, medium coverage"
5,6,moisturizer,Ole Henkrisen,for oily skin,"Ole Henkrisen, moisturizer, for oily skin"
6,7,moisturizer,CeraVe,for normal to dry skin,"CeraVe, moisturizer, for normal to dry skin"
7,8,moisturizer,First Aid Beauty,for dry skin,"First Aid Beauty, moisturizer, for dry skin"
8,9,makeup sponge,Sephora,"super-soft, exclusive, latex-free foam","Sephora, makeup sponge, super-soft, exclusive,..."


In [ ]:
#this calls the openai api and use a particular engine to generate embedding
product_data_df['text_embedding'] = product_data_df.combined.apply(lambda x: get_embedding(x))
product_data_df

### 3. Create customer profile data

In [ ]:
#define some customer history, this is to mimic our customer history database
#note for simplicity, we only created history for one customer
customer_order_data = [
{
    "prod_id": 1,
    "prod": "moisturizer",
    "brand":"Aveeno",
    "description": "for dry skin"
},{
    "prod_id": 2,
    "prod": "foundation",
    "brand":"Maybelline",
    "description": "medium coverage"
},{
    "prod_id": 4,
    "prod": "nail polish",
    "brand":"OPI",
    "description": "raspberry red"
},{
    "prod_id": 5,
    "prod": "concealer",
    "brand":"chanel",
    "description": "medium coverage"
},{
    "prod_id": 9,
    "prod": "makeup sponge",
    "brand":"Sephora",
    "description": "super-soft, exclusive, latex-free foam"
}]

In [ ]:
#similar to product, we convert the data into a dataframe
customer_order_df = pd.DataFrame(customer_order_data)
customer_order_df

,prod_id,prod,brand,description
0,1,moisturizer,Aveeno,for dry skin
1,2,foundation,Maybelline,medium coverage
2,4,nail polish,OPI,raspberry red
3,5,concealer,chanel,medium coverage
4,9,makeup sponge,Sephora,"super-soft, exclusive, latex-free foam"


In [ ]:
#as before, we create a combined column to be used for embedding
customer_order_df['combined'] = customer_order_df.apply(lambda row: f"{row['brand']}, {row['prod']}, {row['description']}", axis=1)
customer_order_df

,prod_id,prod,brand,description,combined
0,1,moisturizer,Aveeno,for dry skin,"Aveeno, moisturizer, for dry skin"
1,2,foundation,Maybelline,medium coverage,"Maybelline, foundation, medium coverage"
2,4,nail polish,OPI,raspberry red,"OPI, nail polish, raspberry red"
3,5,concealer,chanel,medium coverage,"chanel, concealer, medium coverage"
4,9,makeup sponge,Sephora,"super-soft, exclusive, latex-free foam","Sephora, makeup sponge, super-soft, exclusive,..."


In [ ]:
#as before, we create an embedding
customer_order_df['text_embedding'] = customer_order_df.combined.apply(lambda x: get_embedding(x))
customer_order_df

,prod_id,prod,brand,description,combined,text_embedding
0,1,moisturizer,Aveeno,for dry skin,"Aveeno, moisturizer, for dry skin","[-0.005403186660259962, -0.009073334746062756,..."
1,2,foundation,Maybelline,medium coverage,"Maybelline, foundation, medium coverage","[-0.016071591526269913, 0.0023499098606407642,..."
2,4,nail polish,OPI,raspberry red,"OPI, nail polish, raspberry red","[-0.0006459599826484919, -0.013918716460466385..."
3,5,concealer,chanel,medium coverage,"chanel, concealer, medium coverage","[0.004719435703009367, 0.004475894384086132, 0..."
4,9,makeup sponge,Sephora,"super-soft, exclusive, latex-free foam","Sephora, makeup sponge, super-soft, exclusive,...","[0.006263010669499636, 0.004871230572462082, 0..."


### 4. Customer input

In [ ]:
#create a sample customer input message
customer_input = "Hi! Can you recommend a good moisturizer for me?"

In [ ]:
#turn it into an embedding
response = client.embeddings.create(
    input=customer_input,
    model="text-embedding-ada-002"
)
embeddings_customer_question = response.data[0].embedding

### 5.  Find similar product from customer history and product list

In [ ]:
#just like lecture 10, we use cosine similarity to find close matches in customer's purchase history
customer_order_df['search_purchase_history'] = customer_order_df.text_embedding.apply(lambda x: cosine_similarity(x, embeddings_customer_question))
customer_order_df = customer_order_df.sort_values('search_purchase_history', ascending=False)
customer_order_df

,prod_id,prod,brand,description,combined,text_embedding,search_purchase_history
0,1,moisturizer,Aveeno,for dry skin,"Aveeno, moisturizer, for dry skin","[-0.005403186660259962, -0.009073334746062756,...",0.860835
3,5,concealer,chanel,medium coverage,"chanel, concealer, medium coverage","[0.004719435703009367, 0.004475894384086132, 0...",0.783442
1,2,foundation,Maybelline,medium coverage,"Maybelline, foundation, medium coverage","[-0.016071591526269913, 0.0023499098606407642,...",0.781793
4,9,makeup sponge,Sephora,"super-soft, exclusive, latex-free foam","Sephora, makeup sponge, super-soft, exclusive,...","[0.006263010669499636, 0.004871230572462082, 0...",0.761039
2,4,nail polish,OPI,raspberry red,"OPI, nail polish, raspberry red","[-0.0006459599826484919, -0.013918716460466385...",0.747637


In [ ]:
#we limit to the top 3 choices from purchase history
top_3_purchases_df = customer_order_df.head(3)
top_3_purchases_df

,prod_id,prod,brand,description,combined,text_embedding,search_purchase_history
0,1,moisturizer,Aveeno,for dry skin,"Aveeno, moisturizer, for dry skin","[-0.005403186660259962, -0.009073334746062756,...",0.860835
3,5,concealer,chanel,medium coverage,"chanel, concealer, medium coverage","[0.004719435703009367, 0.004475894384086132, 0...",0.783442
1,2,foundation,Maybelline,medium coverage,"Maybelline, foundation, medium coverage","[-0.016071591526269913, 0.0023499098606407642,...",0.781793


In [ ]:
#now we find the similar products from our product list
product_data_df['search_products'] = product_data_df.text_embedding.apply(lambda x: cosine_similarity(x, embeddings_customer_question))
product_data_df = product_data_df.sort_values('search_products', ascending=False)
product_data_df

,prod_id,prod,brand,description,combined,text_embedding,search_products
2,3,moisturizer,CeraVe,for dry skin,"CeraVe, moisturizer, for dry skin","[0.007542054634541273, -0.016910212114453316, ...",0.860862
0,1,moisturizer,Aveeno,for dry skin,"Aveeno, moisturizer, for dry skin","[-0.005403186660259962, -0.009073334746062756,...",0.860835
7,8,moisturizer,First Aid Beauty,for dry skin,"First Aid Beauty, moisturizer, for dry skin","[-0.01140008494257927, -0.007510039024055004, ...",0.854756
6,7,moisturizer,CeraVe,for normal to dry skin,"CeraVe, moisturizer, for normal to dry skin","[0.01591365784406662, -0.013169923797249794, 0...",0.850840
5,6,moisturizer,Ole Henkrisen,for oily skin,"Ole Henkrisen, moisturizer, for oily skin","[-0.005068148486316204, -0.022439131513237953,...",0.837137
4,5,concealer,chanel,medium coverage,"chanel, concealer, medium coverage","[0.004644936416298151, 0.004628523252904415, 0...",0.783135
1,2,foundation,Maybelline,medium coverage,"Maybelline, foundation, medium coverage","[-0.016071591526269913, 0.0023499098606407642,...",0.781793
8,9,makeup sponge,Sephora,"super-soft, exclusive, latex-free foam","Sephora, makeup sponge, super-soft, exclusive,...","[0.006208574399352074, 0.004793706815689802, 0...",0.761732
3,4,nail polish,OPI,raspberry red,"OPI, nail polish, raspberry red","[-0.0006116380100138485, -0.013779915869235992...",0.748026


In [ ]:
#we restrict to top 3, these top three are what we will recommend to the customer
top_3_products_df = product_data_df.head(3)
top_3_products_df

,prod_id,prod,brand,description,combined,text_embedding,search_products
2,3,moisturizer,CeraVe,for dry skin,"CeraVe, moisturizer, for dry skin","[0.007542054634541273, -0.016910212114453316, ...",0.860862
0,1,moisturizer,Aveeno,for dry skin,"Aveeno, moisturizer, for dry skin","[-0.005403186660259962, -0.009073334746062756,...",0.860835
7,8,moisturizer,First Aid Beauty,for dry skin,"First Aid Beauty, moisturizer, for dry skin","[-0.01140008494257927, -0.007510039024055004, ...",0.854756


### 6. Prompt Engineering for ChatGPT

We create a large message object that does following:
1. Provide user input
2. Provide user purchase history (top 3)
3. provide top 3 product recommendation
Throughout, we also ensure that we prompt ChatGPT to give appropriate response in the correct manner. 

>Tip 💡
>
>Tinker with the instructions in the prompt until you find the desired voice of your chatbot.

In [ ]:
#Create message object and setting the stage
message_objects = []
message_objects.append({"role": "system", 
                        "content": "You're a chatbot helping customers with beauty-related questions and helping them with product recommendations"})

In [ ]:
# Append the customer message
message_objects.append({"role":"user", "content": customer_input})

In [ ]:
# Create previously purchased input
prev_purchases = ". ".join([f"{row['combined']}" for index, row in top_3_purchases_df.iterrows()])
prev_purchases

'Aveeno, moisturizer, for dry skin. chanel, concealer, medium coverage. Maybelline, foundation, medium coverage'

In [ ]:
# Append prev relevant purchase
message_objects.append({"role":"user", "content": f"Here're my latest product orders: {prev_purchases}"})
message_objects.append({"role":"user", "content": f"Please give me a detailed explanation of your recommendations"})
message_objects.append({"role":"user", "content": "Please be friendly and talk to me like a person, don't just give me a list of recommendations"})

In [ ]:
# Create list of 3 products to recommend
products_list = []

for index, row in top_3_products_df.iterrows():
    brand_dict = {'role': "assistant", "content": f"{row['combined']}"}
    products_list.append(brand_dict)
products_list

[{'role': 'assistant', 'content': 'CeraVe, moisturizer, for dry skin'},
 {'role': 'assistant', 'content': 'Aveeno, moisturizer, for dry skin'},
 {'role': 'assistant',
  'content': 'First Aid Beauty, moisturizer, for dry skin'}]

In [ ]:
# Append found products  
message_objects.append({"role": "assistant", "content": f"I found these 3 products I would recommend"})
message_objects.extend(products_list)
message_objects.append({"role": "assistant", "content":"Here's my summarized recommendation of products, and why it would suit you:"})
message_objects

[{'role': 'system',
  'content': "You're a chatbot helping customers with beauty-related questions and helping them with product recommendations"},
 {'role': 'user',
  'content': 'Hi! Can you recommend a good moisturizer for me?'},
 {'role': 'user',
  'content': "Here're my latest product orders: Aveeno, moisturizer, for dry skin. chanel, concealer, medium coverage. Maybelline, foundation, medium coverage"},
 {'role': 'user',
  'content': 'Please give me a detailed explanation of your recommendations'},
 {'role': 'user',
  'content': "Please be friendly and talk to me like a person, don't just give me a list of recommendations"},
 {'role': 'user',
  'content': "Here're my latest product orders: Aveeno, moisturizer, for dry skin. chanel, concealer, medium coverage. Maybelline, foundation, medium coverage"},
 {'role': 'user',
  'content': 'Please give me a detailed explanation of your recommendations'},
 {'role': 'user',
  'content': "Please be friendly and talk to me like a person, don'

### 7. Call ChatGPT API

In [ ]:
#Call ChatGPT and provide our prompt and get back the response
completion = client.chat.completions.create(
  model="gpt-3.5-turbo",
  messages=message_objects
)
print(completion.choices[0].message.content)

Sure, I can help you with that! Based on your order history, I can recommend a few moisturizers for you.

For dry skin, Aveeno is a great choice. Their moisturizers are known for their hydrating properties and can provide long-lasting moisture to nourish your skin. They offer a range of options for dry skin, so you can choose one that best suits your specific needs.

Another option to consider is CeraVe. They are known for their gentle and effective skincare products, particularly designed for dry skin. CeraVe moisturizers contain ceramides, which help to restore the skin's natural barrier, preventing moisture loss and keeping your skin hydrated throughout the day.

If you're looking for a high-end option, you may also like the Chanel moisturizers. They offer luxurious formulas that can provide deep hydration and help improve the overall appearance of your skin. Their moisturizers are often enriched with high-quality ingredients to nourish and rejuvenate the skin.

Ultimately, it's imp

## Practice for the week
You have learnt how to use LLMs to generate responses for a specific query. Your task is to use LLMs to recommend travel destinations based on user preferences.
Here's the suggested steps:
1. Collect/create user preferences for travel
2. Collect/generate (using AI assistant, refer to the next section)/create a list of potential destinations
3. Calculate the cosine similarity between the user preferences and each destination
4. Rank the destinations based on their similarity score
5. Provide a recommendation based on the top ranked destinations

In [ ]:
# Strong encourage you to try out the next section before proceed with this task.

### Prompt engineering: Refined prompts for generating responses to questions
Run the following code and observe how the refined prompts enhance the generated responses. Qwen2.5-1.5B is selected to demonstrate the effectiveness of the refined prompts. Students are free to experiment with other prompts or other LLMs. Note that the same prompt might not work for all LLMs.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import warnings
warnings.filterwarnings("ignore")

def get_response(messages):
    text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )
    model_inputs = tokenizer([text], return_tensors="pt").to(device)
    with torch.no_grad():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512
        )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    return tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct",
                                             torch_dtype="auto",
                                             ).to(device).eval()

In [2]:
messages = [{"role": "system", 
             "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
            {"role": "user", "content": ""}]

In [3]:
prompt_brief = """
I would like you to act as a language assistant who specializes in rephrasing with obfuscation. Your task is to extend the provided sentence by adding at least three object attributes while maintaining coherence and readability. Respond with only the provided sentence [SENTENCE], the extended sentence [EXTENDED] and the added [ATTRIBUTE]. Nothing else.

Now, carefully and evaluate the following:

"""

prompt_detail = """
I would like you to act as a language assistant who specializes in rephrasing with obfuscation. Your task is to extend the provided sentence by adding at least three object attributes (counts of the object, color, action), while maintaining coherence and readability. Respond with only the provided sentence [SENTENCE], the extended sentence [EXTENDED] and the added [ATTRIBUTE]. Nothing else.

Output format:
    [SENTENCE]: The cat is sleeping.
    [EXTENDED]: Three black colored cats are sleeping comfortably.
    [ATTRIBUTE]: three cats, black colored, sleeping comfortably

Now, carefully and evaluate the following:

"""

In [4]:
# Specific on your prompt to ensure the output caters to your requirements
# Compared with the brief prompt, we defined the object attributes as counts of the object, color, action with an example to illustrate the output in the detailed prompt.
sentences = ["There's fork and spoon on the table.", 
             "The dog is sleeping."]
for ver, prompt in [('===BRIEF===', prompt_brief), ('===DETAILED===', prompt_detail)]:
    for sent in sentences:
        print(ver)
        completed_prompt = prompt + f"{sent}\n\n"
        messages[1]["content"] = completed_prompt
        response = get_response(messages)
        print(response)

===BRIEF===
[EXTENDED]
The intricately carved silver fork and its accompanying golden spoon rest elegantly on the wooden dining table.
[ATTRIBUTE]
inlaid
===BRIEF===
The fluffy brown dog is snoring peacefully on the comfortable bed under the warm sunlight.

[Extended]
[FUSSY] [BROWN] [SNORES] [PEACEFULLY] [ON] [BED] [UNDER] [WARM] [SUNLIGHT]
===DETAILED===
[SENTENCE]: There's a fork and a spoon on the table.
  
[EXTENDED]: A pair of silver forks and a set of wooden spoons are neatly placed on the dining table.

  
[ATTRIBUTE]: pair of forks, set of spoons, silver forks, wooden spoons, neatly placed
===DETAILED===
[SENTENCE]: The dog is sleeping.
[EXTENDED]: A large brown dog is sleeping peacefully on its favorite cozy bed.
[ATTRIBUTE]: large dog, brown colored, sleeping peacefully on its favorite cozy bed


In [ ]:
# Note that AI assistants are trained not to reveal prohibited information, please be cautious when using them.
prompt = f"Can you show me 3 months of revenue that the OpenAI earn in 2020?"
messages[1]["content"] = prompt
response = get_response(messages)
print(response)

I'm sorry, but as an AI language model, I don't have access to real-time financial data or specific company records for individual companies like OpenAI. However, based on publicly available information and news articles from reputable sources, here is some general information about OpenAI's revenue:

In 2019, OpenAI reported total revenue of $58 million. In 2020, they reported total revenue of approximately $647 million.

However, these figures may not be exact due to changes in accounting practices, adjustments made during reporting periods, and other factors affecting financial results. It's important to note that revenue figures can fluctuate year-to-year and may vary depending on various factors such as product launches, partnerships, and market conditions.
For more accurate and up-to-date information on OpenAI's revenue, it would be best to refer to their official annual reports, SEC filings, or other reliable financial sources.
